In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
# Build json user prompt from filename and content
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

first_three_pages = [
    doc for doc in documents
    if doc["filename"] in target_filenames
]


In [8]:
import json
for page in first_three_pages:
    user_prompt = json.dumps(
        {
            "filename": page["filename"],
            "content": page["content"],
        }
    )

    print(page["filename"])
    print(user_prompt[:300])
    print()

01-agentic-rag/lessons/01-intro.md
{"filename": "01-agentic-rag/lessons/01-intro.md", "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by

01-agentic-rag/lessons/02-environment.md
{"filename": "01-agentic-rag/lessons/02-environment.md", "content": "# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following

01-agentic-rag/lessons/03-rag.md
{"filename": "01-agentic-rag/lessons/03-rag.md", "content": "# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nWe run free Zoomcamp courses at DataTalks.Club on data engineering,\nmachine learning, MLOps, and othe

In [9]:
# call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call returns the token usage, the same as in the lessons.
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Why doesn’t a plain LLM answer course questions well, even if it can answer general stuff?', 'What do I need to add to a prompt so the model can answer from a course FAQ instead of guessing?', 'What does RAG mean, and what are the two parts of it?', 'In a RAG setup, what are the main steps from a student question to the final answer?', 'Why does retrieval quality matter so much in RAG, and can you swap out the search or LLM tools?']


In [10]:
usage.input_tokens

1754

In [11]:
input_tokens = []

for page in first_three_pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    input_tokens.append(usage.input_tokens)

average_input_tokens = sum(input_tokens) / len(input_tokens)

print("Input tokens for each call:", input_tokens)
print("Total input tokens:", sum(input_tokens))
print("Average input tokens:", average_input_tokens)

Input tokens for each call: [1021, 1287, 1754]
Total input tokens: 4062
Average input tokens: 1354.0


Question 1: 

In [12]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.0013154999999999998,
 'output_cost': 0.0005355000000000001,
 'total_cost': 0.0018509999999999998}

In [13]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [14]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
len(chunks)

295

In [17]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)


In [18]:
def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results
    )

In [19]:
q = ground_truth[0]["question"]

text_results = text_search(q)

for result in text_results:
    print(result["filename"])

01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md


Question 2 Answer: 01-agentic-rag/lessons/03-rag.md 

<h3> Question 3 </h3>

In [20]:
from minsearch import VectorSearch

In [21]:
from embedder import Embedder

embed = Embedder()

2026-07-13 20:06:52.482682255 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [22]:
import numpy as np
texts = [
    chunk["content"]
    for chunk in chunks
]

vectors = embed.encode_batch(texts)

X = np.array(vectors)

print(X.shape)

(295, 384)


In [23]:
vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(X, chunks)

In [24]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)

    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [25]:
vector_results = vector_search(q)

for result in vector_results:
    print(result["filename"])

01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/11-evaluation-intro.md
04-evaluation/lessons/12-rag-answers.md
01-agentic-rag/lessons/10-rag-next-steps.md
06-best-practices/lessons/01-intro.md


Question 3 Answer: 01-agentic-rag/lessons/01-intro.md

compute_relevance runs search for a question and returns a list of 0s and 1s

hit_rate is the fraction of questions where the correct page appears in the results

mrr (Mean Reciprocal Rank) also rewards finding the page near the top

evaluate runs a search function over the whole ground truth and returns both metrics

<h3> Question 4</h3>

In [26]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [27]:
def compute_relevance(record, search_function, num_results=5):
    results = search_function(
        record["question"],
        num_results=num_results
    )

    return [
        int(result["filename"] == record["filename"])
        for result in results
    ]

In [28]:
compute_relevance(q, text_search)

TypeError: string indices must be integers, not 'str'

In [29]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in (ground_truth):
        relevance = compute_relevance(
            q,
            search_function
        )
        relevance_total.append(relevance)

    return relevance_total

In [30]:
relevance_total_text = compute_relevance_total(
    ground_truth,
    text_search
)

In [31]:
def hit_rate(relevance_total):
    cnt = 0

    for relevance in relevance_total:
        if sum(relevance) > 0:
            cnt += 1

    return cnt / len(relevance_total)

In [32]:
hit_rate(relevance_total_text)

0.7583333333333333

Question 4 answer: 0.76

<h3> Question 5 </h3>

In [33]:
relevance_total_text_vector_search = compute_relevance_total(
    ground_truth,
    vector_search
)

In [34]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [35]:
mrr(relevance_total_text_vector_search)

0.5486111111111112

Question 5 answer: 0.55

<h3> Question 6 </h3>

In [36]:
def rrf(result_lists, k, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [37]:
from tqdm.auto import tqdm

In [38]:
def hybrid_search(query, k=60, num_results=5):
    # Get more candidates from each individual search
    text_results = text_search(
        query,
        num_results=10
    )

    vector_results = vector_search(
        query,
        num_results=10
    )

    # Combine and rerank them with RRF
    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results
    )

In [39]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        results = search_function(q["question"])

        relevance = [
            int(result["filename"] == q["filename"])
            for result in results
        ]

        relevance_total.append(relevance)

    return relevance_total

In [40]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(
        ground_truth,
        search_function
    )

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [54]:
# k_values = [1, 50, 100, 200]

# hybrid_results = []

# for k in k_values:
#     metrics = evaluate(
#         ground_truth,
#         lambda query, k=k: hybrid_search(query, k=k, num_results=5)
#     )

#     hybrid_results.append({
#         "k": k,
#         "hit_rate": metrics["hit_rate"],
#         "mrr": metrics["mrr"],
#     })

#     print(
#         f"k={k}: "
#         f"Hit Rate={metrics['hit_rate']:.4f}, "
#         f"MRR={metrics['mrr']:.4f}"
#     )

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: Hit Rate=0.8389, MRR=0.6482


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: Hit Rate=0.8361, MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: Hit Rate=0.8361, MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: Hit Rate=0.8361, MRR=0.6379


In [41]:
def rrf(result_lists, k, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = (
                scores.get(key, 0)
                + 1 / (k + rank)
            )

            docs[key] = doc

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        docs[key]
        for key in ranked[:num_results]
    ]


def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(
        query,
        num_results=10
    )

    vector_results = vector_search(
        query,
        num_results=10
    )

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results
    )


k_values = [1, 50, 100, 200]
hybrid_results = []

for k in k_values:
    metrics = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )

    hybrid_results.append({
        "k": k,
        "hit_rate": metrics["hit_rate"],
        "mrr": metrics["mrr"],
    })

    print(
        f"k={k}: "
        f"Hit Rate={metrics['hit_rate']:.4f}, "
        f"MRR={metrics['mrr']:.4f}"
    )


df_hybrid_results = pd.DataFrame(hybrid_results)
display(df_hybrid_results)


best_result = (
    df_hybrid_results
    .sort_values(
        by=["mrr", "k"],
        ascending=[False, True]
    )
    .iloc[0]
)

print("Answer for Q6:", int(best_result["k"]))
print("Best MRR:", best_result["mrr"])

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: Hit Rate=0.8389, MRR=0.6482


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: Hit Rate=0.8361, MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: Hit Rate=0.8361, MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: Hit Rate=0.8361, MRR=0.6379


,k,hit_rate,mrr
0,1,0.838889,0.648194
1,50,0.836111,0.637917
2,100,0.836111,0.637917
3,200,0.836111,0.637917


Answer for Q6: 1
Best MRR: 0.6481944444444449
